# Pipeline de Pré-processamento e Preparação de Dados

Este notebook consolida a limpeza de dados iniciada na EDA (Issue #4) e prepara os conjuntos de treino, validação e teste para o treinamento dos modelos de classificação (Issue #5).

## Objetivos:
1. Realizar o tratamento de valores nulos.
2. Converter o alvo categórico para numérico.
3. Escalonar os atributos físicos.
4. Dividir os dados em proporção 70/15/15.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configurações
pd.set_option('display.max_columns', None)

## 1. Carga dos Dados
Utilizaremos a URL oficial para garantir a consistência com a etapa anterior.

In [ ]:
url = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv'
df_raw = pd.read_csv(url)
print(f"Dataset carregado: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas")

## 2. Seleção de Atributos e Limpeza Inicial
Identificamos na EDA que algumas colunas são totalmente nulas ou são apenas IDs/metadados que não contribuem para a física do problema.

In [ ]:
# Definindo o alvo
target = 'koi_pdisposition'

# Colunas para remover (IDs e metadados irrelevantes para treinamento)
cols_to_drop = [
    'kepid', 'kepoi_name', 'kepler_name', 'koi_disposition', 
    'koi_tce_delivname', 'koi_fittype', 'ra_str', 'dec_str'
]

# Identificando colunas com mais de 50% de valores nulos (conforme observado na EDA)
null_threshold = 0.5 * len(df_raw)
high_null_cols = df_raw.columns[df_raw.isnull().sum() > null_threshold].tolist()

total_drop = list(set(cols_to_drop + high_null_cols))
df = df_raw.drop(columns=total_drop)

print(f"Colunas removidas: {len(total_drop)}")
print(f"Novo formato: {df.shape}")

## 3. Encoding e Tratamento de Nulos
Transformaremos o alvo em binário (1 e 0) e aplicaremos a mediana nos valores ausentes dos atributos numéricos.

In [ ]:
# Encoding do alvo
le = LabelEncoder()
df[target] = le.fit_transform(df[target])
print(f"Classes mapeadas: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Imputação pela mediana em colunas numéricas
df_numeric = df.select_dtypes(include=[np.number])
df_final = df_numeric.fillna(df_numeric.median())

print(f"Nulos remanescentes: {df_final.isnull().sum().sum()}")

## 4. Divisão dos Dados (Split)
Dividiremos em 70% Treino, 15% Validação e 15% Teste.

In [ ]:
X = df_final.drop(columns=[target])
y = df_final[target]

# Primeiro split: Treino vs Resto (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Segundo split: Validação vs Teste (50% do Resto = 15% cada)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Treino: {X_train.shape}")
print(f"Validação: {X_val.shape}")
print(f"Teste: {X_test.shape}")

## 5. Escalonamento (Scaling)
Padronização é vital para o bom desempenho da MLP.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Escalonamento concluído com StandardScaler.")

## 6. Treinamento do Modelo Baseline (Issue #6)
Utilizaremos a **Regressão Logística** como nosso modelo base (benchmark).

In [ ]:
# Instanciando o modelo
baseline = LogisticRegression(max_iter=1000, random_state=42)

# Treinamento
baseline.fit(X_train_scaled, y_train)

print("Modelo Baseline (Regressão Logística) treinado!")

## 7. Avaliação das Métricas
Vamos verificar o desempenho no conjunto de validação para ter uma ideia inicial.

In [ ]:
y_pred = baseline.predict(X_val_scaled)

print("--- Relatório de Classificação (Validação) ---")
print(classification_report(y_val, y_pred, target_names=le.classes_))

print("--- Matriz de Confusão ---")
print(confusion_matrix(y_val, y_pred))

### Conclusão Parcial
O modelo baseline servirá como a métrica mínima a ser batida pela nossa Rede Neural MLP. Se a MLP não superar estes resultados de forma significativa, o custo computacional adicional pode não se justificar.